# Day 3 · 구조가 달라도 같은 입력에 같은 결과가 나옵니다

**여는 법:** 위 메뉴 `런타임 → 모두 실행`. 세션이 끊기면 첫 Cell부터 다시 실행합니다.
**기대 결과:** 세 버전 모두 카페라떼 2잔 학생 주문의 최종 금액이 **7200원**입니다.
**질문:** 결과가 같다면 무엇이 다를까요. 마지막 Cell의 질문에 답을 적습니다.

In [ ]:
# ── 첫 Cell · 준비 ──────────────────────────────────────────
# Colab은 구글이 빌려주는 컴퓨터입니다. 새로 켤 때마다 빈 컴퓨터이므로
# 필요한 파일을 GitHub에서 받아 와야 합니다. 이 Cell이 그 일을 합니다.
#
# 실행하는 법: 이 Cell을 클릭한 뒤 왼쪽 ▶ 버튼을 누르거나 Shift+Enter.
# 위 메뉴 [런타임 → 모두 실행]을 누르면 위에서 아래로 전부 실행됩니다.

import os, sys          # import = 파이썬에 이미 들어 있는 도구 상자를 꺼내는 일

# 줄 앞의 ! 는 "파이썬이 아니라 터미널 명령"이라는 표시입니다.
# git clone = GitHub에 있는 폴더를 통째로 이 컴퓨터로 복사하는 명령.
if not os.path.isdir("jnu-llmops-precourse-day2"):      # 이미 받았으면 건너뜁니다
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 파이썬이 import 할 파일을 찾는 폴더 목록에 어제의 solution 폴더를 넣습니다.
# 이 줄이 없으면 아래 Cell의 from order import Order 가 파일을 못 찾습니다.
sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")

## 버전 A · 네 겹 — 어제의 완료본
`Order` 안에 `items`, 그 안에 `OrderItem`, 그 안에 `quantity`. 계산과 영수증은 다른 파일이 맡습니다.

In [ ]:
# 어제 완료본의 네 파일에서 이름 넷을 꺼내 옵니다.
# from 파일이름 import 이름  = "그 파일 안의 이 이름을 여기서 쓰겠다"
from catalog import MENU               # 메뉴판 (이름 → 가격)
from order import Order                # 주문 상자를 만드는 틀
from pricing import calculate_bill     # 주문 → 합계·할인·최종 금액
from receipt import format_receipt     # 주문 + 금액 → 영수증 문자열

order_a = Order()                                    # 빈 주문 하나
order_a.add("카페라떼", 2, MENU["카페라떼"])          # 항목 추가 (메뉴, 수량, 단가)
bill_a = calculate_bill(order_a, True)               # True = 학생 할인
print(format_receipt(order_a, bill_a))               # 영수증을 화면에 찍는다
total_a = bill_a.total                               # 최종 금액을 이름에 남겨 둔다

## 버전 B · 두 겹 — `OrderItem` 없이 묶음으로
항목을 이름 없는 묶음 `(메뉴, 수량, 단가)`로 담습니다. 계산은 이 Cell 안에서 직접 합니다.

In [ ]:
class OrderB:
    def __init__(self):
        self.items = []                      # (메뉴, 수량, 단가)
    def add(self, menu_name, quantity, unit_price):
        self.items.append((menu_name, quantity, unit_price))

order_b = OrderB()
order_b.add("카페라떼", 2, 4000)
subtotal_b = sum(q * p for _, q, p in order_b.items)
total_b = subtotal_b - subtotal_b * 10 // 100      # 학생 할인 10%
print("수량:", order_b.items[0][1])
print("최종 금액:", total_b, "원")

## 버전 C · 한 덩어리 — 메뉴판·주문·계산·영수증을 `Cafe` 하나에

In [ ]:
class Cafe:
    MENU = {"아메리카노": 3000, "카페라떼": 4000, "초코라떼": 4500}
    def __init__(self):
        self.items = []                      # (메뉴, 수량)
    def add(self, menu_name, quantity):
        self.items.append((menu_name, quantity))
    def total(self, is_student):
        subtotal = sum(q * self.MENU[m] for m, q in self.items)
        discount = subtotal * 10 // 100 if is_student else 0
        return subtotal - discount
    def receipt(self, is_student):
        lines = [f"{m} x {q}" for m, q in self.items]
        lines.append(f"최종 금액: {self.total(is_student)}원")
        return "\n".join(lines)

cafe = Cafe()
cafe.add("카페라떼", 2)
print(cafe.receipt(True))
total_c = cafe.total(True)

## 세 결과를 나란히 놓습니다

In [ ]:
print("A 네 겹   :", total_a)
print("B 두 겹   :", total_b)
print("C 한 덩어리:", total_c)
assert total_a == total_b == total_c == 7200
print("세 구조, 같은 결과")

## 질문 — 답을 이 Cell에 적습니다

1. 학생 할인을 10%에서 15%로 바꾸려면 A·B·C 각각 **어느 줄**을 고쳐야 합니까?
2. 수량에 0을 넣으면 A·B·C는 각각 어디서 멈추거나, 멈추지 않습니까?
3. 세 버전 중 “바뀌는 이유가 하나”인 블록은 어디에 있습니까?

답:
- 1.
- 2.
- 3.